In [2]:
import sys
import os

print("python:", sys.executable)
print("Exixts:", os.path.exists(sys.executable))

python: c:\Projects\Spark_practice\Spark_course\.venv\Scripts\python.exe
Exixts: True


In [3]:
import sys
import os

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(sys.executable)

c:\Projects\Spark_practice\Spark_course\.venv\Scripts\python.exe


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print(sc.version)

3.5.6


In [4]:
# ============================================================
# QUESTION 1 — CUSTOMER TRANSACTION SUMMARY
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# An e-commerce company receives transaction records from multiple stores.
# The analytics team wants a customer-level transaction summary.
 

transactions = sc.parallelize([
    ("T001", "C101", 1200.0),
    ("T002", "C102", 500.0),
    ("T003", "C101", 800.0),
    ("T004", "C103", 1500.0),
    ("T005", "C102", 700.0),
    ("T006", "C101", 1000.0),
    ("T007", "C104", 400.0),
    ("T008", "C103", 500.0)
], 4)


In [7]:
pair_rdd = transactions.map(lambda x:(x[1],(x[2],1)))
result = pair_rdd.reduceByKey(lambda x,y:(x[0]+y[0],x[1]+x[1]))
mapValues = result.mapValues(lambda x:(x[0],x[1],x[0]/x[1]))
mapValues.collect()

[('C103', (2000.0, 2, 1000.0)),
 ('C101', (3000.0, 4, 750.0)),
 ('C104', (400.0, 1, 400.0)),
 ('C102', (1200.0, 2, 600.0))]

In [14]:
# using aggregate by Key
maping = transactions.map(lambda x:(x[1],x[2]))
def add_values(so_far,value):
    return(
        so_far[0]+value,
        so_far[1]+1
    )
    
def combine(tot1,tot2):
    total = tot1[0]+tot2[0]
    count = tot1[1]+tot2[1]
    return(total,count)
    
pair_rdd = maping.aggregateByKey(
    (0,0),
    add_values,
    combine
)      



In [15]:
result = pair_rdd.mapValues(lambda x:(x[0],x[1],x[0]/x[1]))
result.collect()

[('C103', (2000.0, 2, 1000.0)),
 ('C101', (3000.0, 3, 1000.0)),
 ('C104', (400.0, 1, 400.0)),
 ('C102', (1200.0, 2, 600.0))]

In [16]:
# ============================================================
# QUESTION 2 — FAILED TRANSACTION DETECTION
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# A payment company wants customers having repeated failed transactions.

transactions = sc.parallelize([
    ("T001", "C101", "SUCCESS"),
    ("T002", "C102", "FAILED"),
    ("T003", "C101", "FAILED"),
    ("T004", "C103", "SUCCESS"),
    ("T005", "C102", "FAILED"),
    ("T006", "C101", "FAILED"),
    ("T007", "C104", "FAILED"),
    ("T008", "C102", "SUCCESS"),
    ("T009", "C105", "FAILED"),
    ("T010", "C105", "FAILED")
])


In [20]:
filtering = transactions.filter(lambda x:x[2]=="FAILED").map(lambda x:(x[1],1))
result = filtering.reduceByKey(lambda x,y:x+y).filter(lambda x:x[1] >1)
result.collect()



[('C102', 2), ('C105', 2), ('C101', 2)]

In [22]:
# ============================================================
# QUESTION 3 — DUPLICATE TRANSACTION DETECTION
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# Because of retries from an upstream system, some transaction IDs were received multiple times.

transactions = sc.parallelize([
    ("T001", "C101", 500),
    ("T002", "C102", 700),
    ("T001", "C101", 500),
    ("T003", "C103", 900),
    ("T004", "C104", 200),
    ("T002", "C102", 700),
    ("T002", "C102", 700),
    ("T005", "C105", 1000)
])

# Requirement:
# Find duplicate transaction IDs and occurrence count.


In [24]:
filtering = transactions.map(lambda x:(x[0],1))
result = filtering.reduceByKey(lambda x,y:x+y).filter(lambda x:x[1] >1)
result.collect()

[('T001', 2), ('T002', 3)]

In [26]:
# ============================================================
# QUESTION 4 — LATEST CUSTOMER TRANSACTION
# LEVEL: MEDIUM-HARD
# ============================================================

# Scenario:
# Customers can perform multiple transactions.
# Business wants only the latest transaction for every customer.

transactions = sc.parallelize([
    ("C101", "T001", "2026-08-20 10:00:00", 500),
    ("C102", "T002", "2026-08-20 11:00:00", 700),
    ("C101", "T003", "2026-08-21 09:00:00", 900),
    ("C103", "T004", "2026-08-20 12:30:00", 400),
    ("C102", "T005", "2026-08-22 10:15:00", 1200),
    ("C101", "T006", "2026-08-22 15:00:00", 1500)
])


In [27]:
maping = transactions.map(lambda x:(x[0],(x[1],x[2],x[3])))
result = maping.reduceByKey(lambda x,y: x if x[1] > y[1] else y)
result.collect()

[('C102', ('T005', '2026-08-22 10:15:00', 1200)),
 ('C103', ('T004', '2026-08-20 12:30:00', 400)),
 ('C101', ('T006', '2026-08-22 15:00:00', 1500))]

In [34]:
# ============================================================
# QUESTION 5 — PRODUCT REVENUE
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# An online retailer wants total revenue generated by every product.

sales = sc.parallelize([
    ("T001", "Laptop", 2, 60000),
    ("T002", "Mouse", 5, 500),
    ("T003", "Laptop", 1, 60000),
    ("T004", "Keyboard", 3, 1500),
    ("T005", "Mouse", 10, 500),
    ("T006", "Monitor", 2, 12000),
    ("T007", "Keyboard", 2, 1500)
])


In [46]:
maping = sales.map(lambda x:(x[1],x[2]*x[3])).reduceByKey(lambda x,y:x+y)
maping.collect()

    

[('Laptop', 180000), ('Monitor', 24000), ('Mouse', 7500), ('Keyboard', 7500)]

In [49]:
# ============================================================
# QUESTION 6 — CUSTOMER TRANSACTION ENRICHMENT
# LEVEL: MEDIUM
# ============================================================

# Scenario:
# Customer master and transaction data exist separately.
# Transactions must be enriched with customer information.

customers = sc.parallelize([
    ("C101", ("Anuj", "Mumbai")),
    ("C102", ("Rahul", "Delhi")),
    ("C103", ("Priya", "Pune")),
    ("C104", ("Neha", "Bangalore"))
])

transactions = sc.parallelize([
    ("C101", ("T001", 500)),
    ("C102", ("T002", 700)),
    ("C101", ("T003", 900)),
    ("C103", ("T004", 400))
])


In [50]:
result = customers.join(transactions)
result.collect()

[('C103', (('Priya', 'Pune'), ('T004', 400))),
 ('C101', (('Anuj', 'Mumbai'), ('T001', 500))),
 ('C101', (('Anuj', 'Mumbai'), ('T003', 900))),
 ('C102', (('Rahul', 'Delhi'), ('T002', 700)))]

In [57]:
def faltten(records):
    cust_id,(names,trans) = records
    
    name,city = names
    tans_id,amount = trans
    
    return [
        ( tans_id,
        cust_id,
        name,
        city,
        amount)
    ]    
    
result1 = result.flatMap(faltten)
result1.collect()     

[('T004', 'C103', 'Priya', 'Pune', 400),
 ('T001', 'C101', 'Anuj', 'Mumbai', 500),
 ('T003', 'C101', 'Anuj', 'Mumbai', 900),
 ('T002', 'C102', 'Rahul', 'Delhi', 700)]

In [70]:
# ============================================================
# QUESTION 7 — CUSTOMERS WITHOUT TRANSACTIONS
# LEVEL: MEDIUM
# ============================================================

customers = sc.parallelize([
    ("C101", "Anuj"),
    ("C102", "Rahul"),
    ("C103", "Priya"),
    ("C104", "Neha"),
    ("C105", "Amit")
])

transactions = sc.parallelize([
    ("C101", 500),
    ("C102", 700),
    ("C101", 900),
    ("C104", 400)
])

# Requirement:
# Find customers having zero transactions.


In [75]:
join = customers.leftOuterJoin(transactions).filter(lambda x:x[1][1] is None)
join.collect()

[('C103', ('Priya', None)), ('C105', ('Amit', None))]

In [88]:
# ============================================================
# QUESTION 8 — TRANSACTION VALIDATION
# LEVEL: MEDIUM
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 500),
    ("T002", "", 700),
    ("T003", "C103", -100),
    ("", "C104", 900),
    ("T005", "C105", 1200),
    ("T006", "C106", 0)
])

# Validation Rules:
# transaction_id must not be empty
# customer_id must not be empty
# amount > 0

In [91]:
def validation(records):
    
    trans_id = records[0]
    cust_id = records[1]
    amount = records[2]
    
    if trans_id == "":
        return (trans_id, cust_id, amount, "TRANSACTION_ID_MISSING")

    elif cust_id == "":
        return (trans_id, cust_id, amount, "CUSTOMER_ID_MISSING")

    elif amount <= 0:
        return (trans_id, cust_id, amount, "INVALID_AMOUNT")

    else:
        return (trans_id, cust_id, amount, "Valid")
      

In [92]:
rejected_rdd = (
    transactions
    .map(validation)
    .filter(lambda x: x[3] != "Valid")
)

rejected_rdd.collect()

[('T002', '', 700, 'CUSTOMER_ID_MISSING'),
 ('T003', 'C103', -100, 'INVALID_AMOUNT'),
 ('', 'C104', 900, 'TRANSACTION_ID_MISSING'),
 ('T006', 'C106', 0, 'INVALID_AMOUNT')]

In [93]:
# ============================================================
# QUESTION 10 — MULTIPLE VALIDATION FAILURES
# LEVEL: HARD
# ============================================================

transactions = sc.parallelize([
    ("T001", "C101", 500),
    ("", "", -100),
    ("T003", "", 0),
    ("", "C104", 900)
])

# Requirement:
# Return ALL validation errors for each invalid record.


In [94]:
def validation(records):
    trans_id = records[0]
    cust_id = records[1]
    amount = records[2]
    
    errors = []
    
    if trans_id == "":
        errors.append("Missing_transaction_id")
    if cust_id == "":
        errors.append("Missing_customer_id")   
    if amount <= 0:
        errors.append("Missing amount") 
    if errors:
        return(
            trans_id,cust_id,amount,errors
        )    
           
    return None

In [95]:
rejected_rdd = (
    transactions
    .map(validation)
    .filter(lambda x: x is not None)
)

print(rejected_rdd.collect())

[('', '', -100, ['Missing_transaction_id', 'Missing_customer_id', 'Missing amount']), ('T003', '', 0, ['Missing_customer_id', 'Missing amount']), ('', 'C104', 900, ['Missing_transaction_id'])]


In [5]:
# ============================================================
# QUESTION 11 — DAILY REVENUE
# LEVEL: MEDIUM
# ============================================================

transactions = sc.parallelize([
    ("T001", "2026-08-20 10:10:00", 500),
    ("T002", "2026-08-20 11:30:00", 700),
    ("T003", "2026-08-21 09:10:00", 900),
    ("T004", "2026-08-21 12:00:00", 400),
    ("T005", "2026-08-21 15:20:00", 1200),
    ("T006", "2026-08-22 10:00:00", 600)
])

# Requirement:
# Calculate total revenue by transaction date.

In [104]:
pair_rdd = transactions.map(lambda x:(x[1][:10],x[2])).reduceByKey(lambda x,y:x+y)
pair_rdd.collect()

[('2026-08-20', 1200), ('2026-08-21', 2500), ('2026-08-22', 600)]

In [ ]:

# ============================================================
# QUESTION 12 — DAILY TRANSACTION STATISTICS
# LEVEL: HARD
# ============================================================

# Use Question 11 input.

# Requirement:
# For every date calculate:
# transaction_count
# total_amount
# minimum_amount
# maximum_amount
# average_amount

In [8]:
def seq_func(acc,amount):
    count = acc[0]+1
    total = acc[1]+amount
    minimum = min(acc[2],amount)
    maximum = max(acc[3],amount)
    
    return (count,total,minimum,maximum)

def comb_func(acc1,acc2):
    count = acc1[0]+acc2[0]
    total = acc1[1]+acc2[1]
    minimum = min(acc1[2],acc2[2])
    maximum = max(acc1[3],acc2[3])
    
    return (count,total,minimum,maximum)

In [12]:
pair_rdd = transactions.map(lambda x: (x[1][:10],x[2]))
result = pair_rdd.aggregateByKey(
    (0,0,float('inf'),float('-inf')),
    seq_func,
    comb_func    
)
final_result = result.mapValues(
    lambda x: (
        x[0],                    # count
        x[1],                    # total
        x[2],                    # minimum
        x[3],                    # maximum
        round(x[1] / x[0], 2)    # average
    )
)   
final_result.collect() 

[('2026-08-20', (2, 1200, 500, 700, 600.0)),
 ('2026-08-21', (3, 2500, 400, 1200, 833.33)),
 ('2026-08-22', (1, 600, 600, 600, 600.0))]

In [13]:
# ============================================================
# QUESTION 16 — MOST POPULAR PRODUCT PER CUSTOMER
# LEVEL: HARD
# ============================================================

transactions = sc.parallelize([
    ("C101", "Laptop"),
    ("C101", "Mouse"),
    ("C101", "Laptop"),
    ("C102", "Keyboard"),
    ("C102", "Mouse"),
    ("C102", "Mouse"),
    ("C103", "Monitor"),
    ("C103", "Monitor"),
    ("C103", "Laptop")
])

In [17]:
maping = transactions.map(lambda x:((x[0],x[1]),1))
result = maping.reduceByKey(lambda x,y:x+y)
result.collect()

[(('C101', 'Mouse'), 1),
 (('C103', 'Monitor'), 2),
 (('C103', 'Laptop'), 1),
 (('C101', 'Laptop'), 2),
 (('C102', 'Keyboard'), 1),
 (('C102', 'Mouse'), 2)]

In [18]:
final_result = result.map(lambda x:(x[0][0],(x[0][1],x[1]))).reduceByKey(lambda x,y:(x if x[1]>y[1] else y))
final_result.collect()

[('C103', ('Monitor', 2)), ('C102', ('Mouse', 2)), ('C101', ('Laptop', 2))]

In [30]:
# ============================================================
# QUESTION 20 — LATEST RECORD FROM HISTORICAL + INCREMENTAL
# LEVEL: HARD
# ============================================================

historical = sc.parallelize([
    ("C101", ("Mumbai", "2026-08-20")),
    ("C102", ("Delhi", "2026-08-20")),
    ("C103", ("Pune", "2026-08-20"))
])

incremental = sc.parallelize([
    ("C101", ("Bangalore", "2026-08-22")),
    ("C103", ("Hyderabad", "2026-08-21")),
    ("C104", ("Chennai", "2026-08-22"))
])

In [31]:
pair_rdd = historical.union(incremental)
result = pair_rdd.reduceByKey(lambda x,y:(x if x[1]> y[1] else y))
result.collect()

[('C103', ('Hyderabad', '2026-08-21')),
 ('C101', ('Bangalore', '2026-08-22')),
 ('C104', ('Chennai', '2026-08-22')),
 ('C102', ('Delhi', '2026-08-20'))]

In [41]:
# ============================================================
# QUESTION 22 — MALFORMED CSV RECORDS
# LEVEL: HARD
# ============================================================

raw_rdd = sc.parallelize([
    "T001,C101,500.50,SUCCESS",
    "T002,C102,ABC,FAILED",
    "T003,C103,900.25,SUCCESS",
    "T004,C104",
    "T005,C105,1200,SUCCESS"
])

# Validation Rules:
# - Exactly 4 columns
# - Amount must be numeric


In [46]:
def valid_parser(records):
    
    columns = records.split(",")
    
    if len(columns) != 4:
        return(False,records)
    
    try:
        amount = float(columns[2])
    except ValueError:
        return (False, records)   
    
    return (
        True,(
        columns[0],
        columns[1],
        amount,
        columns[3]
        )
    ) 
    

In [47]:
validated_rdd = raw_rdd.map(valid_parser)

valid_rdd = validated_rdd.filter(
    lambda x: x[0]
).map(
    lambda x: x[1]
)


invalid_rdd = validated_rdd.filter(
    lambda x: not x[0]
).map(
    lambda x: x[1]
)

print("Valid:", valid_rdd.collect())
print("Invalid:", invalid_rdd.collect())



Valid: [('T001', 'C101', 500.5, 'SUCCESS'), ('T003', 'C103', 900.25, 'SUCCESS'), ('T005', 'C105', 1200.0, 'SUCCESS')]
Invalid: ['T002,C102,ABC,FAILED', 'T004,C104']


In [48]:
# ============================================================
# QUESTION 26 — CITY LEVEL REVENUE
# LEVEL: MEDIUM-HARD
# ============================================================

customers = sc.parallelize([
    ("C101", "Mumbai"),
    ("C102", "Delhi"),
    ("C103", "Mumbai"),
    ("C104", "Pune")
])

transactions = sc.parallelize([
    ("C101", 500),
    ("C102", 700),
    ("C101", 900),
    ("C103", 400),
    ("C104", 1200),
    ("C103", 600)
])

In [52]:
reduce = transactions.reduceByKey(lambda x,y:x+y)
pair = reduce.join(customers)
pair.collect()
result = pair.map(lambda x: (x[1][1],x[1][0])).reduceByKey(lambda x,y:x+y)
result.collect()



[('Pune', 1200), ('Mumbai', 2400), ('Delhi', 700)]

In [56]:
customers = sc.parallelize([
    ("C101", "Anuj"),
    ("C102", "Rahul"),
    ("C103", "Priya")
])

transactions = sc.parallelize([
    ("C101", ("T001", 500)),
    ("C999", ("T002", 700)),
    ("C102", ("T003", 900)),
    ("C888", ("T004", 400))
])

In [57]:
result = transactions.subtractByKey(customers)
result.collect()

[('C888', ('T004', 400)), ('C999', ('T002', 700))]

In [58]:
# ============================================================
# QUESTION 29 — HIGHEST TRANSACTION PER CUSTOMER
# LEVEL: HARD
# ============================================================

transactions = sc.parallelize([
    ("C101", ("T001", 500)),
    ("C101", ("T002", 900)),
    ("C101", ("T003", 700)),
    ("C102", ("T004", 400)),
    ("C102", ("T005", 1200)),
    ("C103", ("T006", 800))
])

# Requirement:
# Find highest-value transaction per customer.

In [63]:
result = (transactions.reduceByKey(lambda x,y: (x if x[1]>y[1] else y)))
result.collect()

[('C102', ('T005', 1200)), ('C103', ('T006', 800)), ('C101', ('T002', 900))]

In [71]:
# ============================================================
# QUESTION 30 — FRAUD DETECTION
# LEVEL: HARD
# ============================================================
transactions = sc.parallelize([
    ("C101", 1500),
    ("C101", 2000),
    ("C101", 2500),

    ("C102", 500),
    ("C102", 400),
    ("C102", 600),
    ("C102", 700),

    ("C103", 2000),
    ("C103", 1000),

    ("C104", 6000)
])

# Rules:
# Flag customer when:
# total transaction amount > 5000
# OR
# transaction_count >= 4

In [72]:
pair = transactions.map(
    lambda x: (x[0], (x[1], 1))
)

result = pair.reduceByKey(
    lambda x, y: (
        x[0] + y[0],   # total amount
        x[1] + y[1]    # transaction count
    )
)


# Filter flagged customers and add fraud reason
final_result = result.filter(
    lambda x: x[1][0] > 5000 or x[1][1] >= 4
).map(
    lambda x: (
        x[0],             # customer_id
        x[1][1],          # transaction_count
        x[1][0],          # total_amount
        "HIGH_TOTAL_AMOUNT"
        if x[1][0] > 5000
        else "HIGH_TRANSACTION_FREQUENCY"
    )
)

print(final_result.collect())
    

[('C102', 4, 2200, 'HIGH_TRANSACTION_FREQUENCY'), ('C101', 3, 6000, 'HIGH_TOTAL_AMOUNT'), ('C104', 1, 6000, 'HIGH_TOTAL_AMOUNT')]


In [73]:
# ============================================================
# QUESTION 35 — USER JOURNEY
# LEVEL: HARD
# ============================================================

events = sc.parallelize([
    ("U101", 1, "LOGIN"),
    ("U101", 3, "PURCHASE"),
    ("U101", 2, "SEARCH"),
    ("U102", 2, "SEARCH"),
    ("U102", 1, "LOGIN"),
    ("U102", 3, "LOGOUT")
])

# Requirement:
# Reconstruct event sequence for each user.

In [77]:
pair = events.map(lambda x:(x[0],x[2])).groupByKey()
for custmer, prod in pair.collect():
    print ((custmer ,list(prod)))

('U101', ['LOGIN', 'PURCHASE', 'SEARCH'])
('U102', ['SEARCH', 'LOGIN', 'LOGOUT'])


In [93]:
# ============================================================
# QUESTION 36 — FIRST AND LAST TRANSACTION
# LEVEL: HARD
# ============================================================

transactions = sc.parallelize([
    ("C101", "T001", "2026-08-20 10:00"),
    ("C101", "T002", "2026-08-22 11:00"),
    ("C101", "T003", "2026-08-21 09:00"),
    ("C102", "T004", "2026-08-20 12:00"),
    ("C102", "T005", "2026-08-23 14:00")
])

# Requirement:
# Find:
# first_transaction_id
# last_transaction_id
# for each customer.

In [96]:
pair_rdd = transactions.map(
    lambda x: (x[0], (x[1], x[2]))
)

print(pair_rdd.collect())

def find_first_last(a, b):

    # a = (transaction_id, date)
    # b = (transaction_id, date)

    if a[1] < b[1]:
        first = a
    else:
        first = b

    if a[1] > b[1]:
        last = a
    else:
        last = b

    return (first, last)

result = pair_rdd.reduceByKey(find_first_last)

[('C101', ('T001', '2026-08-20 10:00')), ('C101', ('T002', '2026-08-22 11:00')), ('C101', ('T003', '2026-08-21 09:00')), ('C102', ('T004', '2026-08-20 12:00')), ('C102', ('T005', '2026-08-23 14:00'))]


In [98]:
# ============================================================
# QUESTION 38 — DAILY REVENUE COMPARISON
# LEVEL: HARD
# ============================================================

yesterday = sc.parallelize([
    ("P101", 1000),
    ("P102", 2000),
    ("P103", 1500)
])

today = sc.parallelize([
    ("P101", 1200),
    ("P102", 1800),
    ("P103", 2000),
    ("P104", 500)
])

# Requirement:
# Return:
# product_id
# yesterday_revenue
# today_revenue
# difference



In [104]:
pair = yesterday.fullOuterJoin(today)
result = pair.mapValues(lambda x:(x[0] or 0,x[1] or 0,(x[1] or 0)-(x[0] or 0)))
result.collect()

[('P104', (0, 500, 500)),
 ('P102', (2000, 1800, -200)),
 ('P103', (1500, 2000, 500)),
 ('P101', (1000, 1200, 200))]

In [122]:
# ============================================================
# QUESTION 41 — SOURCE VS TARGET DATA RECONCILIATION
# LEVEL: HARD
# ============================================================

source = sc.parallelize([
    ("T001", "C101", 500),
    ("T002", "C102", 700),
    ("T003", "C103", 900),
    ("T004", "C104", 400),
    ("T005", "C105", 1200)
])

target = sc.parallelize([
    ("T001", "C101", 500),
    ("T002", "C102", 700),
    ("T003", "C103", 950),
    ("T005", "C105", 1200),
    ("T006", "C106", 300)
])

# Requirement:
# Create:
# missing_in_target
# extra_in_target
# data_mismatch

In [125]:
source_pair = source.map(lambda x:(x[0],(x[1],x[2])))
target_pair = target.map(lambda x:(x[0],(x[1],x[2])))

missing_in_target = source_pair.subtractByKey(target_pair)
missing_in_target.collect()


[('T004', ('C104', 400))]

In [126]:
extra_in_target = target_pair.subtractByKey(source_pair)
extra_in_target.collect()

[('T006', ('C106', 300))]

In [127]:
data_mismatch = (
    source_pair.join(target_pair)
    .filter(lambda x: x[1][0] != x[1][1])
    .map(lambda x:(x[0],x[1][0],x[1][1]))
)
data_mismatch.collect()

[('T003', ('C103', 900), ('C103', 950))]

In [130]:
# ============================================================
# QUESTION 42 — DEDUPLICATION USING LATEST TIMESTAMP
# LEVEL: HARD
# ============================================================

customers = sc.parallelize([
    ("C101", "Anuj", "Mumbai", "2026-08-20 10:00"),
    ("C101", "Anuj", "Delhi", "2026-08-21 11:00"),
    ("C102", "Rahul", "Pune", "2026-08-20 09:00"),
    ("C102", "Rahul", "Mumbai", "2026-08-22 12:00"),
    ("C103", "Priya", "Delhi", "2026-08-20 13:00")
])

# Requirement:
# Keep latest customer version.

In [132]:
pair_rdd = customers.map(lambda x:(x[0],(x[1],x[2],x[3])))

def latest_rec(a,b):
    if a[2] > b[2]:
        return a
    else:
        return b
    
result = pair_rdd.reduceByKey(latest_rec)

result.collect()    


[('C102', ('Rahul', 'Mumbai', '2026-08-22 12:00')),
 ('C103', ('Priya', 'Delhi', '2026-08-20 13:00')),
 ('C101', ('Anuj', 'Delhi', '2026-08-21 11:00'))]

In [6]:
# ============================================================
# QUESTION 43 — SCD TYPE 1 USING RDD
# LEVEL: HARD
# ============================================================

existing = sc.parallelize([
    ("C101", "Anuj", "Mumbai"),
    ("C102", "Rahul", "Delhi"),
    ("C103", "Priya", "Pune")
])

incoming = sc.parallelize([
    ("C101", "Anuj", "Bangalore"),
    ("C103", "Priya", "Hyderabad"),
    ("C104", "Neha", "Chennai")
])

# Requirement:
# Changed existing record -> overwrite
# Unchanged existing record -> retain
# New customer -> insert

In [10]:
result = (
    existing
    .map(lambda x: (x[0], (x[1], x[2])))
    .fullOuterJoin(
        incoming.map(lambda x: (x[0], (x[1], x[2])))
    )
    .map(lambda x: (
        x[0],
        *(x[1][1] if x[1][1] is not None else x[1][0])
    ))
)

result.collect()

[('C101', 'Anuj', 'Bangalore'),
 ('C102', 'Rahul', 'Delhi'),
 ('C103', 'Priya', 'Hyderabad'),
 ('C104', 'Neha', 'Chennai')]

In [5]:
# ============================================================
# QUESTION 45 — TOP PRODUCT PER CATEGORY
# LEVEL: HARD
# ===========================================================

sales = sc.parallelize([
    ("Electronics", "Laptop", 5000),
    ("Electronics", "Mouse", 1000),
    ("Electronics", "Monitor", 3000),
    ("Fashion", "Shirt", 1500),
    ("Fashion", "Jeans", 2500),
    ("Fashion", "Shoes", 2000),
    ("Grocery", "Rice", 800),
    ("Grocery", "Oil", 1200)
])

# Requirement:
# Find highest-revenue product per category.


In [6]:
result = (sales
        .map(lambda x: (x[0],(x[1],x[2])))
        .reduceByKey(lambda x,y: x if x[1] > y[1] else y)
)
result.collect()

[('Grocery', ('Oil', 1200)),
 ('Electronics', ('Laptop', 5000)),
 ('Fashion', ('Jeans', 2500))]

In [14]:
# ============================================================
# QUESTION 47 — DETECT DATA SKEW
# LEVEL: HARD
# ============================================================

transactions = sc.parallelize(
    [("C101", i) for i in range(1, 101)] +
    [("C102", i) for i in range(1, 6)] +
    [("C103", i) for i in range(1, 4)] +
    [("C104", 1)]
)

# Requirement:
# Calculate record count per customer.
# Flag customers where count > 50.

In [15]:
result = transactions.map(lambda x:(x[0],1)).reduceByKey(lambda x,y:x+y).filter(lambda x:x[1]>50)
result.collect()

[('C101', 100)]

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 1679)
Traceback (most recent call last):
  File "C:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 755, in __init__
    self.handle()
  File "c:\Projects\Spark_practice\Spark_course\.venv\Lib\site-packages\pyspark\accumulators.py", line 295, in handle
    poll(accum_updates)
  File "c:\Projects\Spark_practice\Spark_course\.venv\Lib\site-packages\pyspark\